In [1]:
load('dat_feat_2015.Rdata')

In [1]:
load('RA_label.Rdata')

In [3]:
load('CHF_label.Rdata')

In [5]:
load('Depression_label.Rdata')

In [ ]:
dat.label

In [1]:
# RA is PheCode 714.1, which means Rheumatoid Arthritis(风湿性关节炎)

In [2]:
name_desc <- read.csv('https://han-attention.s3.amazonaws.com/new_task/data813/name_desc/name_desc_all_GPT4.csv')
emb_all <- read.csv('https://han-attention.s3.amazonaws.com/new_task/data813/embedding/h%5ETBh_code_emb.csv')
emb_MGB <- emb_all[1:6917,]

In [4]:
library(purrr)

gen_pai_emb <- function(emb, dat.feat, phecode) {
  colnames(dat.feat) <- gsub('CCS-PCS', 'CCS', colnames(dat.feat))
  matching_cols <- which(colnames(dat.feat) %in% emb[,1])
  dat.feat <- dat.feat[,c(1, matching_cols, ncol(dat.feat))]
  emb <- emb[match(colnames(dat.feat)[2:(ncol(dat.feat)-1)], emb[,1]),]
  emb_phe <- as.matrix(emb[match(phecode, emb[,1]), 2:769])
  emb <- t(apply(emb[,2:769],1,as.numeric))
  cos <- emb %*% t(emb_phe)

  compute_emb_row <- function(row) {
    freq_code <- as.numeric(row[2:(ncol(dat.feat) - 1)]) * cos
    return(t(emb) %*% freq_code)
  }

  # Initialize txtProgressBar
  pb <- txtProgressBar(min = 0, max = nrow(dat.feat), style = 3)
  emb_pat <- matrix(, nrow = nrow(dat.feat), ncol = ncol(emb))

  for(i in 1:nrow(dat.feat)) {
    emb_pat[i, ] <- compute_emb_row(as.numeric(dat.feat[i, ]))
    setTxtProgressBar(pb, i)  # Update progress bar
  }

  close(pb)  # Close the progress bar

  emb_pat <- t(apply(emb_pat, 2, function(x) x/sqrt(sum(x^2))))
  return(emb_pat)
}


In [ ]:
dat.label

In [7]:
which(dat.label[,2] == "Y")

[1]  20  23  29  36  44  84 164 176 236 261 279 312 350 358 362 423 438 490 492
[20] 501 510 513 554 578 592 638 656 659 674 675 698 701 725 750 753 791 811 876
[39] 881 893 912 922 926 935 943 944 959 978

In [ ]:
emb_pat <- gen_pai_emb(emb_MGB, dat.feat[1:], 'PheCode:714.1')

  |                                                                      |   0%

In [108]:
rank_visualization<-function(data,group,test)
{
  mylevel<-unique(group)
  r<-length(mylevel)
  
  pos<-vector("list",r)
  
  for(i in 1:r)
  {
    pos[[i]]<-(group==mylevel[i])
  }
  
  
  Z<-data
  N<-dim(data)[1]
  
  D<-as.matrix(dist(Z,method = "euclidean"))
  RG<-matrix(rep(0,N^2),nrow = N)
  
  for(i in 1:N)
  {
    RG[i,]<-rank(D[i,])
  }
  
  library(MASS)
  
  M<-matrix(NA,nrow=N,ncol=r)
  
  for(i in 1:N)
  {
    for(j in 1:r)
    {
      a<-RG[pos[[j]],i]
      M[i,j]<-mean(a[a!=0])
    }
  }
  
  lw<-dim(test)[1]
  MW<-matrix(NA,nrow = lw,ncol = r)
  for(k in 1:lw)
  {
    w<-test[k,]
    
    dw<-rep(0,N)
    
    for(i in 1:N)
    {
      dw[i]<-norm(w-data[i,],type="2")
    }
    
    rw<-rep(0,N)
    
    for(i in 1:N)
    {
      rw[i]<-sum(D[i,]<dw[i])+0.5
    }
    
    Mw<-rep(NA,r)
    
    for(i in 1:r)
    {
      Mw[i]<-mean(rw[pos[[i]]])
    }
    
    MW[k,]<-Mw
  }
  
  ans <- list(M,MW)
  names(ans) <- c("train_k_dimension","test_k_dimension")
  return(ans)
}

In [ ]:
prepare_data <- function(emb_pat, dat.feat, dat.label, Y_num, N_num){
    group <- dat.label[,2]
    Y <- which(group == "Y")
    N <- which(group == "N")
    Y_train <- sample(Y, num_Y)
    N_train <- sample(N, num_N)
}